# Mind the Gap: BERT training on Kaggle

Before running:
1. **Settings → Accelerator → GPU T4 x2** (or P100).
2. **Settings → Internet → On** (needed to download `bert-base-uncased`; requires a verified phone number on Kaggle).
3. **Add Input** → your dataset `mind-the-gap-upload` (the zip with `src/` and `data/processed/`).

Then **Run All**. At the end, download `bert_results.zip` from the Output panel.

In [ ]:
import glob, os, shutil, subprocess
# Find the uploaded folder that contains src/ and data/processed/
candidates = glob.glob('/kaggle/input/**/src/train_bert.py', recursive=True)
assert candidates, 'Upload not found: check Add Input'
upload_root = os.path.dirname(os.path.dirname(candidates[0]))
print('uploaded project:', upload_root)

# Copy to a writable folder
shutil.copytree(upload_root, '/kaggle/working/project', dirs_exist_ok=True)
os.chdir('/kaggle/working/project')
print(os.listdir('.'), os.listdir('data/processed'))

In [ ]:
!pip install -q peft
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable the GPU in Settings')

## 1. Smoke test (1-2 minutes)
Pretrained BERT on 200 windows for 2 epochs: checks download, GPU and that the loss goes down.

In [ ]:
!python -m src.train_bert --signal resp --subset 200 --epochs 2

## 2. Real training, one model per signal
Early stopping on validation MAE. Only the validation set is predicted here.

In [ ]:
!python -m src.train_bert --signal hr

In [ ]:
!python -m src.train_bert --signal resp

In [ ]:
!python -m src.train_bert --signal stress

## 3. Download the results
Copy `results/predictions/bert_*` and `results/bert/` into the same folders of your repo, then run `python -m src.evaluate` locally.

In [ ]:
!zip -qr /kaggle/working/bert_results.zip results/bert results/predictions/bert_*
print('Download bert_results.zip from the Output panel')